# Machine Learning System Design: Core Architecture Patterns

Designing large-scale ML systems requires decoupling concerns and applying proven patterns:
1. **Multi-Stage Cascade Ranking Pattern**: Filtering millions of candidates through progressively heavier models (Retrieval -> Scoring -> Reranking).
2. **Asynchronous Event-Driven Decoupling**: Offloading long-running ML tasks to queue-worker architectures (Kafka / RabbitMQ pattern).
3. **Dual-Store Architecture (Lambda / Kappa for ML)**: Real-time low-latency feature serving coupled with offline historical analytical storage.

In [ ]:
import sys
from pathlib import Path
for p in [Path.cwd() / 'code', Path('08-system-design/architecture-patterns/code')]:
    if p.exists():
        sys.path.insert(0, str(p.resolve()))
import patterns
print('patterns loaded')

## 1. Multi-Stage Cascade Ranking Pattern
Funneling millions of items down to a personalized top-K list within strict latency SLAs.

In [ ]:
from patterns import CandidateItem, CandidateRetriever, FineRanker, DiversityReranker

# 1. Catalog of items (representing a large repository of content)
catalog = [
    CandidateItem("item_101", "tech", popularity_score=0.92, relevance_features=[0.9, 0.8]),
    CandidateItem("item_102", "tech", popularity_score=0.88, relevance_features=[0.85, 0.75]),
    CandidateItem("item_103", "tech", popularity_score=0.74, relevance_features=[0.7, 0.6]),
    CandidateItem("item_104", "gaming", popularity_score=0.95, relevance_features=[0.6, 0.9]),
    CandidateItem("item_105", "gaming", popularity_score=0.89, relevance_features=[0.55, 0.85]),
    CandidateItem("item_106", "finance", popularity_score=0.65, relevance_features=[0.2, 0.3])
]

# Stage 1: Candidate Retrieval (Coarse filter: select preferred categories)
retriever = CandidateRetriever(catalog)
candidates = retriever.retrieve(preferred_categories={"tech", "gaming"}, top_k=4)
print(f"Stage 1 (Retrieval): Filtered catalog to {len(candidates)} candidates.")

# Stage 2: Fine-grained Ranking (User interaction scoring)
ranker = FineRanker()
user_profile_features = [0.95, 0.85]
scored_candidates = ranker.score_candidates(user_profile_features, candidates)
print("\nStage 2 (Fine Ranking):")
for item, score in scored_candidates:
    print(f" - [{item.category:6s}] {item.item_id}: score={score:.4f}")

# Stage 3: Diversity Reranking (Enforce max 2 per category, pick top 3)
final_recommendations = DiversityReranker.rerank(scored_candidates, max_per_category=2, final_k=3)
print("\nStage 3 (Final Diversified Output):")
for item, score in final_recommendations:
    print(f" -> [{item.category:6s}] {item.item_id} (Final Score={score:.4f})")

## 2. Asynchronous Event-Driven Inference Queue
Decoupling heavy model inference from synchronous HTTP request/response lifecycles.

In [ ]:
from patterns import AsyncInferenceQueue

# Model worker function
def heavy_nlp_inference(payload):
    text = payload.get("document_text", "")
    return {"entity_count": len(text.split()), "sentiment": "POSITIVE"}

queue_engine = AsyncInferenceQueue(worker_fn=heavy_nlp_inference)

# 1. API Gateway accepts request and returns 202 Accepted with task_id
task = queue_engine.submit_task("req_8921", {"document_text": "Production machine learning systems require scalable architecture."})
print(f"Task submitted: ID={task.task_id}, Status={task.status}")

# 2. Background worker consumes and computes
finished_task = queue_engine.process_next()
print(f"Worker completed: ID={finished_task.task_id}, Status={finished_task.status}")
print(f"Inference Result: {finished_task.result}")